In [5]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/mrnotalent/laptop-embedding/final_clean_filled_laptops.csv"  
ALPHA = 0.05

df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
print("Columns:", list(df.columns))
print()

results = []


def normality_ok(x, label=""):

    x = pd.Series(x).dropna()
    if len(x) < 20:
        return False, None
    stat, p = stats.normaltest(x)
    return p > ALPHA, p


def levene_ok(*groups):
    groups = [pd.Series(g).dropna() for g in groups]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) < 2:
        return False, None
    stat, p = stats.levene(*groups)
    return p > ALPHA, p


def add_result(hypothesis, variables, test, stat, p, effect_name, effect_val, notes=""):
    decision = "Reject H0" if p < ALPHA else "Fail to reject H0"
    results.append({
        "hypothesis": hypothesis,
        "variables": variables,
        "test": test,
        "statistic": round(float(stat), 4) if stat is not None else None,
        "p_value": p,
        "alpha": ALPHA,
        "decision": decision,
        "effect_size_name": effect_name,
        "effect_size_value": round(float(effect_val), 4) if effect_val is not None else None,
        "notes": notes,
    })
    print(f"[{hypothesis}] {test}: stat={stat:.4f}, p={p:.4g}, "
          f"{effect_name}={effect_val:.4f} -> {decision}")
    print("  ", notes)
    print()



if "category" in df.columns and "price_usd" in df.columns:
    sub = df[["category", "price_usd"]].dropna()
    cats = sub["category"].value_counts()
    keep_cats = cats[cats >= 5].index  
    sub = sub[sub["category"].isin(keep_cats)]
    groups = [g["price_usd"].values for _, g in sub.groupby("category")]
    group_names = [k for k, _ in sub.groupby("category")]

    norm_flags = [normality_ok(g)[0] for g in groups]
    var_ok, var_p = levene_ok(*groups)

    if all(norm_flags) and var_ok:
        stat, p = stats.f_oneway(*groups)
        grand_mean = sub["price_usd"].mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        ss_total = ((sub["price_usd"] - grand_mean) ** 2).sum()
        eta_sq = ss_between / ss_total
        test_name = "One-way ANOVA"
        effect_name, effect_val = "eta_squared", eta_sq
        notes = (f"Groups: {group_names}. Normality/variance assumptions held "
                  f"(Levene p={var_p:.4g}); parametric test used.")
    else:
        stat, p = stats.kruskal(*groups)
        n = len(sub)
        eps_sq = (stat - len(groups) + 1) / (n - len(groups))
        test_name = "Kruskal-Wallis H-test"
        effect_name, effect_val = "epsilon_squared", max(eps_sq, 0)
        notes = (f"Groups: {group_names}. Normality and/or variance-homogeneity "
                  f"assumptions violated (Levene p={var_p}); non-parametric test used.")

    add_result("H1: price differs by category", "price_usd ~ category",
                test_name, stat, p, effect_name, effect_val, notes)

    if p < ALPHA and test_name == "One-way ANOVA":
        import itertools
        try:
            from statsmodels.stats.multicomp import pairwise_tukeyhsd
            tukey = pairwise_tukeyhsd(sub["price_usd"], sub["category"], alpha=ALPHA)
            print("Post-hoc Tukey HSD:\n", tukey, "\n")
        except ImportError:
            print("  (statsmodels not available -- install for Tukey HSD post-hoc)\n")



if "gpu" in df.columns and "price_usd" in df.columns:
    gpu_str = df["gpu"].astype(str).str.lower()
    dedicated_kw = ["nvidia", "geforce", "rtx", "gtx", "radeon", "rx ", "quadro", "amd radeon"]
    integrated_kw = ["intel uhd", "intel iris", "intel hd", "integrated", "vega 8", "vega 3"]

    is_dedicated = gpu_str.str.contains("|".join(dedicated_kw), na=False)
    is_integrated = gpu_str.str.contains("|".join(integrated_kw), na=False) & ~is_dedicated

    sub = df.loc[is_dedicated | is_integrated, ["price_usd"]].copy()
    sub["gpu_type"] = np.where(is_dedicated[sub.index], "dedicated", "integrated")
    sub = sub.dropna()

    ded = sub.loc[sub.gpu_type == "dedicated", "price_usd"]
    integ = sub.loc[sub.gpu_type == "integrated", "price_usd"]

    if len(ded) >= 5 and len(integ) >= 5:
        n1_ok, _ = normality_ok(ded)
        n2_ok, _ = normality_ok(integ)
        var_ok, var_p = levene_ok(ded, integ)

        if n1_ok and n2_ok and var_ok:
            stat, p = stats.ttest_ind(ded, integ, equal_var=True)
            test_name = "Independent t-test"
            pooled_std = np.sqrt(((len(ded)-1)*ded.std()**2 + (len(integ)-1)*integ.std()**2) / (len(ded)+len(integ)-2))
            cohens_d = (ded.mean() - integ.mean()) / pooled_std
            effect_name, effect_val = "cohens_d", cohens_d
            notes = f"n_dedicated={len(ded)}, n_integrated={len(integ)}. Assumptions held; parametric test used."
        else:
            stat, p = stats.mannwhitneyu(ded, integ, alternative="two-sided")
            n1, n2 = len(ded), len(integ)
            effect_name = "rank_biserial_r"
            effect_val = 1 - (2 * stat) / (n1 * n2)
            notes = f"n_dedicated={len(ded)}, n_integrated={len(integ)}. Assumptions violated; non-parametric test used."

        add_result("H2: dedicated GPU vs integrated GPU price", "price_usd ~ gpu_type",
                    test_name, stat, p, effect_name, effect_val, notes)
    else:
        print("H2 skipped: not enough observations in one of the GPU groups.\n")



if "ram_gb" in df.columns and "price_usd" in df.columns:
    sub = df[["ram_gb", "price_usd"]].dropna()
    norm_ram, _ = normality_ok(sub["ram_gb"])
    norm_price, _ = normality_ok(sub["price_usd"])

    if norm_ram and norm_price:
        stat, p = stats.pearsonr(sub["ram_gb"], sub["price_usd"])
        test_name = "Pearson correlation"
        effect_name, effect_val = "r", stat
        notes = f"n={len(sub)}. Both variables approx. normal; Pearson used."
    else:
        stat, p = stats.spearmanr(sub["ram_gb"], sub["price_usd"])
        test_name = "Spearman correlation"
        effect_name, effect_val = "rho", stat
        notes = f"n={len(sub)}. Normality violated; Spearman (rank) used."

    add_result("H3: RAM associated with price", "ram_gb ~ price_usd",
                test_name, stat, p, effect_name, effect_val, notes)



if "gpu" in df.columns and "category" in df.columns:
    gpu_str = df["gpu"].astype(str).str.lower()
    is_dedicated = gpu_str.str.contains("|".join(dedicated_kw), na=False)
    is_integrated = gpu_str.str.contains("|".join(integrated_kw), na=False) & ~is_dedicated
    sub = df.loc[is_dedicated | is_integrated, ["category"]].copy()
    sub["gpu_type"] = np.where(is_dedicated[sub.index], "dedicated", "integrated")

    cat_counts = sub["category"].value_counts()
    sub = sub[sub["category"].isin(cat_counts[cat_counts >= 5].index)]

    if sub["category"].nunique() >= 2:
        contingency = pd.crosstab(sub["category"], sub["gpu_type"])
        stat, p, dof, expected = stats.chi2_contingency(contingency)
        n = contingency.sum().sum()
        min_dim = min(contingency.shape) - 1
        cramers_v = np.sqrt((stat / n) / min_dim) if min_dim > 0 else np.nan

        low_expected = (expected < 5).sum()
        notes = (f"Contingency table shape {contingency.shape}, n={n}. "
                 f"{low_expected} cell(s) with expected count < 5 "
                 f"(chi-square approximation may be unreliable if this is large).")

        add_result("H4: GPU type associated with category", "gpu_type ~ category",
                    "Chi-square test of independence", stat, p, "cramers_v", cramers_v, notes)
        print("Contingency table:\n", contingency, "\n")



src_col = None
for candidate in ["source_dataset", "source", "dataset_source"]:
    if candidate in df.columns:
        src_col = candidate
        break

if src_col and "price_usd" in df.columns:
    sub = df[[src_col, "price_usd"]].dropna()
    src_counts = sub[src_col].value_counts()
    keep = src_counts[src_counts >= 5].index
    sub = sub[sub[src_col].isin(keep)]
    groups = [g["price_usd"].values for _, g in sub.groupby(src_col)]
    group_names = [k for k, _ in sub.groupby(src_col)]

    if len(groups) >= 2:
        norm_flags = [normality_ok(g)[0] for g in groups]
        var_ok, var_p = levene_ok(*groups)

        if all(norm_flags) and var_ok:
            stat, p = stats.f_oneway(*groups)
            grand_mean = sub["price_usd"].mean()
            ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
            ss_total = ((sub["price_usd"] - grand_mean) ** 2).sum()
            eta_sq = ss_between / ss_total
            test_name = "One-way ANOVA"
            effect_name, effect_val = "eta_squared", eta_sq
            notes = f"Sources: {group_names}. Parametric assumptions held."
        else:
            stat, p = stats.kruskal(*groups)
            n = len(sub)
            eps_sq = (stat - len(groups) + 1) / (n - len(groups))
            test_name = "Kruskal-Wallis H-test"
            effect_name, effect_val = "epsilon_squared", max(eps_sq, 0)
            notes = f"Sources: {group_names}. Assumptions violated; non-parametric used."

        add_result("H5: price differs by source dataset", f"price_usd ~ {src_col}",
                    test_name, stat, p, effect_name, effect_val,
                    notes + " Interpret as association only -- source reflects "
                            "differing product mixes/retailers, not a causal price driver.")
else:
    print("H5 skipped: no source-dataset column found "
          "(checked source_dataset/source/dataset_source).\n")


results_df = pd.DataFrame(results)
results_df.to_csv("hypothesis_test_results.csv", index=False)
print("=" * 60)
print("Saved hypothesis_test_results.csv")
print(results_df.to_string(index=False))

Loaded: (1960, 25)
Columns: ['row_uid', 'title', 'price_usd', 'price_original', 'price_original_currency', 'cpu', 'ram_gb', 'storage', 'gpu', 'display', 'battery', 'category', 'document_text', 'has_review_text', 'source_dataset', 'source_collection_date', 'source_row_id', 'has_page_provenance', 'ram_gb_is_imputed', 'battery_is_imputed', 'display_is_imputed', 'cpu_is_imputed', 'storage_is_imputed', 'gpu_is_imputed', 'category_is_imputed']

[H1: price differs by category] Kruskal-Wallis H-test: stat=918.3593, p=1.9e-183, epsilon_squared=0.4707 -> Reject H0
   Groups: ['2 in 1 Convertible', 'Basic Laptops', 'Creative Laptops', 'Gaming', 'Gaming Laptops', 'Home > Computer Systems > Laptop / Notebook > All Laptop > ASUS', 'Home > Computer Systems > Laptop / Notebook > All Laptop > Acer America', 'Home > Computer Systems > Laptop / Notebook > All Laptop > Apple', 'Home > Computer Systems > Laptop / Notebook > All Laptop > DELL', 'Home > Computer Systems > Laptop / Notebook > All Laptop > HP'